# Why is there more matter than antimatter?

*cosmology · advanced* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/cosmology/baryon_asymmetry)

The universe holds about $6\times 10^{-10}$ baryons per photon and almost no antimatter. Sakharov's three conditions say what any explanation needs; leptogenesis is a leading candidate: heavy Majorana neutrinos decay slightly more often into leptons than antileptons (CP asymmetry $\varepsilon$), inverse decays wash part of it out (strength $K$), and sphalerons turn the leptons into baryons. The Boltzmann equations for thousands of $(\varepsilon, K)$ models are solved as one batched ODE and compared with the observed $\eta_B$. An open problem.

**Vector form**

$$\frac{dN_N}{dz} = -D\,(N_N - N_N^{eq}),\qquad \frac{dN_{B-L}}{dz} = \varepsilon\,D\,(N_N - N_N^{eq}) - W\,N_{B-L},\qquad D = Kz\frac{K_1(z)}{K_2(z)},\quad W = \frac{K}{4}z^3K_1(z),\qquad \eta_B = \frac{a_{sph}}{f}\,N_{B-L}^f$$

**Index (tensor) form**

$$\Delta_{k,n+1} = \Delta_{kn}\,e^{-D_kh} - \frac{dN^{eq}}{dz}\,h\,\varphi(D_kh),\qquad N_{ek,n+1} = N_{ekn}\,e^{-W_kh} + \varepsilon_e D_k\,\bar\Delta_k\,h\,\varphi(W_kh),\qquad \varphi(x) = \frac{1 - e^{-x}}{x}$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $z_n$ | `z_grid(z_max, n)` | (T,) | $z = M/T$, log-spaced: the clock runs as the universe cools |
| $\varepsilon_e,\ K_k$ | `parameter_grid(...)` | (E, 1), (1, K) | CP asymmetry as a column, decay parameter as a row: everything broadcasts to $(E, K)$ |
| $N_N^{eq}$ | `n_eq(z)` | (T,) | $\tfrac38 z^2K_2(z)$: $3/4$ per photon while relativistic, then $e^{-z}$ |
| $D_k,\ W_k$ | `decay_rate(z, K), washout(z, K)` | (1, K) | decays and inverse decays per Hubble time at the step midpoint |
| $\Delta_k$ | `delta` | (1, K) | $N_N - N_N^{eq}$: the departure from equilibrium (Sakharov 3) |
| $\varphi(x)$ | `-expm1(-x) / x` | (1, K) | exact one-step factor of a linear ODE: stable even for $K = 1000$ |
| $N_{ek}$ | `n_bl` | (E, K) | $B - L$ asymmetry of every model, advanced in one tensor update per step |
| $\eta_B$ | `A_SPH / DILUTION * n_bl` | (E, K) | sphalerons keep $28/79$ of $B - L$ as baryons; photons grow by $f = 2387/86$ |
| $\kappa_f$ | `n_bl / (0.75 * eps)` | (K,) | efficiency: $1$ for weak washout, $\approx 1/(K z_B)$ for strong washout |

Observations: the baryon-to-photon ratio is $\eta_B = (6.12 \pm 0.04)\times 10^{-10}$ from the CMB (Planck 2018, $\Omega_bh^2 = 0.0224$), in agreement with Big Bang nucleosynthesis, and there is no sign of antimatter galaxies. Starting from a symmetric universe, Sakharov (1967) showed that an asymmetry needs (1) baryon-number violation, (2) C and CP violation and (3) a departure from thermal equilibrium. The tests check (2) and (3) directly: $\varepsilon = 0$, or heavy neutrinos held at equilibrium, give exactly zero; (1) is the Standard Model's sphaleron process, which converts $B - L$ into $B = \tfrac{28}{79}(B - L)$ above about 130 GeV. The CP violation of the quarks (CKM phase) is far too small, and the electroweak transition is a crossover, so the Standard Model alone fails: new physics is needed. Leptogenesis (Fukugita and Yanagida 1986) uses the heavy right-handed neutrinos of the seesaw mechanism, which would also explain why light neutrinos are so light. This toy keeps only decays and inverse decays of the lightest one, $N_1$ (no $\Delta L = 2$ scatterings, flavour effects or thermal corrections), in the normalisation of Buchmüller, Di Bari and Plümacher (2005). Weak washout ($K \ll 1$) with a thermal start keeps the full asymmetry ($\kappa_f \to 1$, checked); strong washout ($K \gg 1$) loses most of it but forgets the initial conditions (the two starts merge), and the solver agrees with their analytic fit to within about 10%. With $K = \tilde m_1/m_*$, $m_* \approx 1.08\times10^{-3}$ eV, the neutrino masses seen in oscillations ($\approx 0.009$–$0.05$ eV) point to $K \approx 10$–$50$, where $\varepsilon \approx 2\times10^{-6}$–$10^{-5}$ is needed; the Davidson–Ibarra bound $\varepsilon \lesssim 10^{-6}\,(M_1/10^{10}\,\mathrm{GeV})$ then requires $M_1 \gtrsim 10^{10}$ GeV. Status: open. Leptogenesis is attractive and consistent with everything known, but it is unconfirmed: the heavy neutrinos are far beyond colliders. Indirect hints would be neutrinoless double-beta decay (Majorana nature) and CP violation in neutrino oscillations (T2K, NOvA, and next DUNE and Hyper-Kamiokande); electroweak baryogenesis and other mechanisms remain alternatives. The page explores the toy equations numerically and does not claim this is how the asymmetry arose.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/cosmology/baryon_asymmetry/main.py
"""Main loop: (ε, K) grid → batched Boltzmann equations → η_B, compared with the observed 6.1 × 10⁻¹⁰."""

from pathlib import Path

import torch

from problems._common.runtime import cli, fast_mode

from . import equation, plot, solve, tensors


def run(params: dict, out_dir: Path) -> Path:
    # CPU, float64: Boltzmann suppression e^{−z} spans tens of decades.
    fast = fast_mode()
    n_steps = 600 if fast else params["steps"]
    z = tensors.z_grid(params["z_max"], n_steps)
    n_eps, n_K = (9, 13) if fast else (params["n_eps"], params["n_K"])
    eps, K = tensors.parameter_grid((-8.0, -4.0), (-2.0, 3.0), n_eps, n_K)

    grid = solve.solve(z, eps, K, initial=params["initial"])
    eta = solve.baryon_to_photon(grid["n_bl_final"])  # (E, K)
    kappa = solve.efficiency(grid["n_bl_final"][0], eps[0, 0])  # (K,) the same for every ε (linear source)
    kappa_zero = solve.efficiency(solve.solve(z, eps[:1], K, initial="zero")["n_bl_final"][0], eps[0, 0])
    print(f"{n_eps}×{n_K} models (ε = 1e-8 … 1e-4, K = 0.01 … 1000), {n_steps} exponential-integrator steps in z = M/T")
    print(
        f"η_B ranges from {eta.min():.2e} to {eta.max():.2e}; observed η_B = {tensors.ETA_OBSERVED:.2e} (Planck, BBN)"
    )
    for k_val in (0.01, 1.0, 10.0, 100.0):
        j = int((K[0] - k_val).abs().argmin())
        need = solve.required_epsilon(kappa[j], tensors.ETA_OBSERVED)
        print(f"K = {K[0, j]:8.3g} (m̃1 = {equation.effective_mass(K[0, j].item()):.1e} eV): κ_f = {kappa[j]:.3e}"
              f" → observed η_B needs ε ≈ {need:.1e}")  # fmt: skip

    eps_pick = 10 ** params["log_eps"]
    K_pick = 10 ** params["log_K"]
    one = solve.solve(z, torch.tensor([[eps_pick]], dtype=torch.float64),
                      torch.tensor([[K_pick]], dtype=torch.float64), initial=params["initial"])  # fmt: skip
    eta_pick = solve.baryon_to_photon(one["n_bl_final"]).item()
    print(f"chosen model ε = {eps_pick:.2e}, K = {K_pick:.3g}: η_B = {eta_pick:.2e} "
          f"({eta_pick / tensors.ETA_OBSERVED:.2f} × observed)")  # fmt: skip
    print("checks: ε = 0 or N_N held at equilibrium give exactly zero asymmetry (see test.py)")
    data = {"eps": eps, "K": K, "eta": eta, "kappa": kappa}
    return plot.animate(data, one, kappa_zero, eps_pick, K_pick, tensors.ETA_OBSERVED, out_dir / "baryon_asymmetry.gif")


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/cosmology/baryon_asymmetry/equation.py
r"""Physics → tensor translation: the Boltzmann equations of thermal leptogenesis (decays and inverse decays).

A heavy Majorana neutrino N of mass M decays into a lepton and a Higgs, slightly more often into one than into the
CP conjugate (CP asymmetry ε). With z = M/T as the clock and abundances per comoving volume that holds one photon
at z ≪ 1 (Buchmüller, Di Bari & Plümacher 2005), the two equations are

    dN_N/dz      = −D(z)\,(N_N − N_N^{eq}),
    dN_{B−L}/dz  = ε\,D(z)\,(N_N − N_N^{eq}) − W(z)\,N_{B−L},

    D = K z \frac{K_1(z)}{K_2(z)},     W = \frac14 K z^3 K_1(z),     N_N^{eq} = \frac38 z^2 K_2(z),

with K_1, K_2 modified Bessel functions. The decay parameter K = Γ_D / H(T = M) compares the decay rate to the
expansion rate: K ≪ 1 is weak washout, K ≫ 1 strong washout. Sphalerons turn the final B − L into baryons:

    η_B = \frac{a_{sph}}{f}\,N_{B−L}^f,   a_{sph} = 28/79,   f = 2387/86 (photon production after leptogenesis),

i.e. η_B ≈ 0.96 × 10^{-2} ε κ_f, where κ_f = N_{B−L}^f / (\tfrac34 ε) is the efficiency factor
(≤ 1 for a thermal start).
Here ε is defined as the B − L produced per decay, so ε > 0 makes matter.
"""

import torch

A_SPH = 28 / 79  # fraction of B − L that sphalerons leave as B (Standard Model, one Higgs doublet)
DILUTION = 2387 / 86  # f: photons per comoving volume grow by this factor until recombination


def k1(z):
    return torch.special.modified_bessel_k1(z)


def k2(z):
    """K_2(z) = K_0(z) + (2/z) K_1(z) (recurrence; torch has K_0 and K_1)."""
    return torch.special.modified_bessel_k0(z) + 2 / z * k1(z)


def n_eq(z: torch.Tensor) -> torch.Tensor:
    r"""N_N^{eq} = (3/8) z² K₂(z): 3/4 while relativistic (z ≪ 1), Boltzmann-suppressed e^{−z} later."""
    return 3 / 8 * z**2 * k2(z)


def dn_eq_dz(z: torch.Tensor) -> torch.Tensor:
    r"""dN^{eq}/dz = −(3/8) z² K₁(z), from d(z²K₂)/dz = −z²K₁."""
    return -3 / 8 * z**2 * k1(z)


def decay_rate(z: torch.Tensor, K: torch.Tensor) -> torch.Tensor:
    r"""D = K z K₁/K₂: decays per Hubble time (K₁/K₂ is the average time-dilation factor ⟨1/γ⟩).   → broadcast"""
    return K * z * k1(z) / k2(z)


def washout(z: torch.Tensor, K: torch.Tensor) -> torch.Tensor:
    r"""W = (1/4) K z³ K₁(z): inverse decays ℓ H → N erase the asymmetry; switched off by e^{−z} once T < M."""
    return K / 4 * z**3 * k1(z)


def phi(x: torch.Tensor) -> torch.Tensor:
    r"""(1 − e^{−x})/x, equal to 1 at x = 0 (the exact one-step factor for dy/dz = −a y + s with constant a, s)."""
    return torch.where(x > 1e-12, -torch.expm1(-x) / x.clamp_min(1e-300), 1 - x / 2)


def baryon_to_photon(n_bl: torch.Tensor) -> torch.Tensor:
    r"""η_B = (a_{sph}/f) N_{B−L}^f ≈ 0.0128 N_{B−L}^f."""
    return A_SPH / DILUTION * n_bl


def kappa_strong_washout(K: torch.Tensor) -> torch.Tensor:
    r"""Analytic fit of the efficiency (decays + inverse decays, any start): κ_f = (2/(z_B K))(1 − e^{−z_B K/2})."""
    z_b = 2 + 4 * K**0.13 * torch.exp(-2.5 / K)
    return 2 / (z_b * K) * (1 - torch.exp(-z_b * K / 2))


def effective_mass(K: float) -> float:
    r"""m̃₁ = K m* in eV, with the equilibrium neutrino mass m* ≈ 1.08 × 10⁻³ eV."""
    return K * M_STAR_EV


M_STAR_EV = 1.08e-3


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/cosmology/baryon_asymmetry/tensors.py
"""Build the concrete tensors: the z = M/T clock and the (ε, K) grid of models solved together."""

import torch

ETA_OBSERVED = 6.12e-10  # baryons per photon: Planck 2018 Ω_b h² = 0.0224 (η = 273.9e-10 Ω_b h²), BBN agrees
Z_START = 0.01  # start of the integration, T = 100 M: the heavy neutrinos are relativistic


def z_grid(z_max: float, n: int, device: torch.device | None = None) -> torch.Tensor:
    """Log-spaced z = M/T from Z_START to z_max (n points): (n,). Equal steps in ln z suit rates that vary as powers."""
    return torch.logspace(torch.log10(torch.tensor(Z_START)), torch.log10(torch.tensor(z_max)), n,
                          dtype=torch.float64, device=device)  # fmt: skip


def parameter_grid(eps_range, K_range, n_eps: int, n_K: int, device: torch.device | None = None):
    """ε as a column (E, 1) and K as a row (1, K), log-spaced, so every rate broadcasts to the (E, K) grid."""
    eps = torch.logspace(*eps_range, n_eps, dtype=torch.float64, device=device)[:, None]
    K = torch.logspace(*K_range, n_K, dtype=torch.float64, device=device)[None, :]
    return eps, K


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/cosmology/baryon_asymmetry/solve.py
r"""Integrate the Boltzmann equations for the whole (ε, K) grid at once with an exponential integrator.

For strong washout the rates D and W reach thousands per unit z, so an explicit method would need tiny steps. Both
equations are linear, of the form dy/dz = −a y + s, and over one step with a, s frozen at the midpoint the exact
solution is y_{n+1} = y_n e^{−a h} + s h (1 − e^{−a h})/(a h). This is stable for any step and exact in both limits.
The neutrino equation is written for the departure from equilibrium Δ = N_N − N^{eq}:

    dΔ/dz = −D Δ − dN^{eq}/dz,        dN_{B−L}/dz = ε D Δ − W N_{B−L}.
"""

import torch

from problems._common.progress import steps

from . import equation


def solve(z: torch.Tensor, eps: torch.Tensor, K: torch.Tensor, initial: str = "thermal", equilibrium: bool = False):
    """Abundances along z for every model: eps (E, 1) and K (1, K) broadcast to (E, K).

    initial: "thermal" (N_N = N^eq at the start) or "zero" (no heavy neutrinos yet, produced by inverse decays).
    equilibrium=True holds N_N = N^eq by hand (infinitely fast interactions), the test of Sakharov's third condition.
    Returns N_N (1, K, T), N_{B−L} (E, K, T) and the final N_{B−L} (E, K).
    """
    shape = torch.broadcast_shapes(eps.shape, K.shape)
    delta = torch.zeros_like(K) if initial == "thermal" else -equation.n_eq(z[0]) * torch.ones_like(K)  # (1, K)
    n_bl = torch.zeros(shape, dtype=z.dtype, device=z.device)  # (E, K)
    deltas, n_bls = [delta], [n_bl]
    for i in steps(len(z) - 1, "z steps"):
        h = z[i + 1] - z[i]
        zm = (z[i + 1] * z[i]).sqrt()  # midpoint in ln z
        D, W = equation.decay_rate(zm, K), equation.washout(zm, K)  # (1, K)
        if equilibrium:
            new_delta = torch.zeros_like(delta)
        else:
            new_delta = delta * torch.exp(-D * h) - equation.dn_eq_dz(zm) * h * equation.phi(D * h)
        source = eps * D * (delta + new_delta) / 2  # (E, K): ε D Δ at the midpoint
        n_bl = n_bl * torch.exp(-W * h) + source * h * equation.phi(W * h)
        delta = new_delta
        deltas.append(delta)
        n_bls.append(n_bl)
    n_n = torch.stack(deltas, -1) + equation.n_eq(z)  # (1, K, T)
    n_bl_t = torch.stack(n_bls, -1)  # (E, K, T)
    return {"z": z, "n_n": n_n, "n_bl": n_bl_t, "n_bl_final": n_bl_t[..., -1]}


def efficiency(n_bl_final: torch.Tensor, eps: torch.Tensor) -> torch.Tensor:
    r"""κ_f = N_{B−L}^f / (\tfrac34 ε): the share of the maximal asymmetry that survives washout."""
    return n_bl_final / (0.75 * eps)


def baryon_to_photon(n_bl_final: torch.Tensor) -> torch.Tensor:
    return equation.baryon_to_photon(n_bl_final)


def required_epsilon(kappa: torch.Tensor, eta: float) -> torch.Tensor:
    r"""ε needed to reach η: η = (a_{sph}/f) \tfrac34 ε κ_f solved for ε."""
    return eta / (equation.A_SPH / equation.DILUTION * 0.75 * kappa)


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/cosmology/baryon_asymmetry/plot.py
"""η_B over the (K, ε) plane, the abundances evolving with z = M/T, and the efficiency κ_f(K)."""

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from problems._common.plotting import ACCENT, frame_indices, save_gif

from . import equation


def animate(grid, one, kappa_zero, eps_pick, K_pick, eta_obs, path: Path) -> Path:
    logK = np.log10(grid["K"][0].numpy())
    loge = np.log10(grid["eps"][:, 0].numpy())
    eta = grid["eta"].numpy()  # (E, K)
    z = one["z"].numpy()
    n_n, n_eq = one["n_n"][0, 0].numpy(), equation.n_eq(one["z"]).numpy()
    kappa_z = one["n_bl"][0, 0].numpy() / (0.75 * eps_pick)  # running efficiency N_{B−L}/(¾ε)
    eta_z = equation.baryon_to_photon(one["n_bl"][0, 0]).numpy()

    fig = plt.figure(figsize=(9.4, 4.6))
    gs = fig.add_gridspec(2, 2, width_ratios=[1.05, 1])
    ax = fig.add_subplot(gs[:, 0])
    levels = np.arange(-14, -3.5, 1.0)
    cf = ax.contourf(logK, loge, np.log10(np.clip(eta, 1e-14, None)), levels=levels, cmap="Blues")
    fig.colorbar(cf, ax=ax, label=r"$\log_{10}\eta_B$", shrink=0.85)
    ax.contour(logK, loge, eta, levels=[eta_obs], colors=[ACCENT[1]], linewidths=2)
    ax.plot([], [], color=ACCENT[1], lw=2, label=r"observed $\eta_B = 6.1\times10^{-10}$")
    ax.plot(np.log10(K_pick), np.log10(eps_pick), "o", color=ACCENT[4], ms=8, mec="white", label="model shown right")
    ax.axvline(0, color="#6b7280", lw=0.8, ls=":")
    ax.text(-1.9, loge[-1] - 0.3, "weak washout", fontsize=7, color="white")
    ax.text(0.2, loge[-1] - 0.3, "strong washout", fontsize=7, color="white")
    ax.set_xlabel(r"$\log_{10} K$  (decay rate / expansion rate)")
    ax.set_ylabel(r"$\log_{10}\varepsilon$  (CP asymmetry per decay)")
    ax.legend(fontsize=7, loc="lower right")
    ax.set_title(f"{eta.size} models solved as one batched ODE", fontsize=8.5)

    ax_z = fig.add_subplot(gs[0, 1])
    ax_z.loglog(z, n_eq, color="#6b7280", lw=1, ls="--", label=r"$N_N^{eq}$")
    line_n = ax_z.loglog([], [], color=ACCENT[0], lw=2, label=r"$N_N$ heavy neutrinos")[0]
    line_k = ax_z.loglog([], [], color=ACCENT[4], lw=2, label=r"$N_{B-L}\,/\,(3\varepsilon/4)$")[0]
    cursor = ax_z.axvline(z[0], color="#9ca3af", lw=0.8)
    ax_z.set_xlim(z[0], z[-1])
    ax_z.set_ylim(1e-5, 2)
    ax_z.set_xlabel(r"$z = M/T$ (time →)")
    ax_z.legend(fontsize=6.5, loc="lower left")
    title = ax_z.set_title("", fontsize=8)

    ax_k = fig.add_subplot(gs[1, 1])
    Ks = grid["K"][0].numpy()
    ax_k.loglog(Ks, grid["kappa"].numpy(), color=ACCENT[0], lw=2, label="thermal start (solver)")
    ax_k.loglog(Ks, kappa_zero.numpy(), color=ACCENT[2], lw=1.5, label="zero start (solver)")
    ax_k.loglog(Ks, equation.kappa_strong_washout(grid["K"][0]).numpy(), color="#111827", lw=1, ls="--",
                label="analytic fit (BDP 2005)")  # fmt: skip
    ax_k.axvline(K_pick, color=ACCENT[4], lw=1)
    ax_k.set_xlabel(r"$K$")
    ax_k.set_ylabel(r"efficiency $\kappa_f$")
    ax_k.set_ylim(1e-5, 2)
    ax_k.legend(fontsize=6.5, loc="lower left")

    def update(step: int):
        line_n.set_data(z[: step + 1], n_n[: step + 1])
        k = np.clip(kappa_z[: step + 1], 1e-12, None)
        line_k.set_data(z[: step + 1], k)
        cursor.set_xdata([z[step], z[step]])
        title.set_text(f"ε = {eps_pick:.1e}, K = {K_pick:.3g}: T = M/{z[step]:.2f}, η_B so far = {eta_z[step]:.2e}")

    fig.tight_layout()
    return save_gif(fig, update, frame_indices(len(z), 45), path)


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'log_eps': -5.7,
    'log_K': 1.0,
    'initial': 'thermal',
    'n_eps': 41,
    'n_K': 61,
    'steps': 3000,
    'z_max': 100.0
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.cosmology.baryon_asymmetry")]:
    del sys.modules[name]
main = importlib.import_module("problems.cosmology.baryon_asymmetry.main")
problem_dir = Path("problems/cosmology/baryon_asymmetry")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
